# Notebook 3 - Command Grounding

Roadmap role: Week 3. Literature-driven rule: grounding must use explicit map data and must not silently guess ambiguous locations.

This notebook orchestrates the current deterministic grounding slice. Shared implementation lives in `src/shepherd_ai/grounding.py`. Current evaluations are synthetic and do not measure real-world grounding accuracy.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT / 'shepherd-ai').exists():
    ROOT = ROOT / 'shepherd-ai'
sys.path.insert(0, str(ROOT / 'src'))

MAP_PATH = ROOT / 'datasets' / 'maps' / 'shepherd_test_map_v1.csv'
REGION_GEOJSON_PATH = ROOT / 'datasets' / 'maps' / 'shepherd_test_map_regions_v1.geojson'
EXAMPLES_PATH = ROOT / 'datasets' / 'maps' / 'grounding_examples_v1.jsonl'
DIAGNOSTICS_PATH = ROOT / 'datasets' / 'maps' / 'grounding_diagnostics_v1.jsonl'
HOLDOUT_PATH = ROOT / 'datasets' / 'maps' / 'grounding_holdout_synthetic_v1.jsonl'

print('root', ROOT)
print('map exists', MAP_PATH.exists())
print('region geojson exists', REGION_GEOJSON_PATH.exists())
print('examples exist', EXAMPLES_PATH.exists())
print('diagnostics exist', DIAGNOSTICS_PATH.exists())
print('synthetic holdout exists', HOLDOUT_PATH.exists())

In [ ]:
from shepherd_ai.grounding import ground_intent, load_map_locations
from shepherd_ai.intent import parse_intent

locations = load_map_locations(MAP_PATH)
intent = parse_intent('Send two drones north and inspect the crops.')
grounded = ground_intent(intent, locations)
grounded.to_dict()

In [ ]:
constraint_intent = parse_intent('Inspect the greenhouse and avoid the power lines.')
constraint_grounded = ground_intent(constraint_intent, locations)
constraint_grounded.to_dict()

In [ ]:
region_locations = load_map_locations(REGION_GEOJSON_PATH)
polygon_intent = parse_intent('Inspect the polygon zone.')
polygon_grounded = ground_intent(polygon_intent, region_locations)
polygon_grounded.to_dict()

In [ ]:
!python scripts/validate_map_dataset.py --map datasets/maps/shepherd_test_map_v1.csv --json-output outputs/evaluations/map_validation_shepherd_test_map_v1.json --markdown-output reports/week3_map_validation_report.md

In [ ]:
!python scripts/validate_map_dataset.py --map datasets/maps/shepherd_test_map_regions_v1.geojson --json-output outputs/evaluations/map_validation_shepherd_test_map_regions_v1.json --markdown-output reports/week3_region_geojson_map_validation_report.md

In [ ]:
!python scripts/evaluate_grounding.py --map datasets/maps/shepherd_test_map_v1.csv --dataset datasets/maps/grounding_examples_v1.jsonl --output outputs/evaluations/grounding_examples_v1.json

In [ ]:
!python scripts/validate_grounding_dataset.py --map datasets/maps/shepherd_test_map_v1.csv --dataset datasets/maps/grounding_examples_v1.jsonl --summary-output outputs/evaluations/grounding_examples_v1_validation.json

In [ ]:
!python scripts/evaluate_grounding.py --map datasets/maps/shepherd_test_map_v1.csv --dataset datasets/maps/grounding_diagnostics_v1.jsonl --output outputs/evaluations/grounding_diagnostics_v1.json

In [ ]:
!python scripts/validate_grounding_dataset.py --map datasets/maps/shepherd_test_map_v1.csv --dataset datasets/maps/grounding_diagnostics_v1.jsonl --summary-output outputs/evaluations/grounding_diagnostics_v1_validation.json

In [ ]:
!python scripts/evaluate_grounding.py --map datasets/maps/shepherd_test_map_v1.csv --dataset datasets/maps/grounding_holdout_synthetic_v1.jsonl --output outputs/evaluations/grounding_holdout_synthetic_v1.json

In [ ]:
!python scripts/validate_grounding_dataset.py --map datasets/maps/shepherd_test_map_v1.csv --dataset datasets/maps/grounding_holdout_synthetic_v1.jsonl --summary-output outputs/evaluations/grounding_holdout_synthetic_v1_validation.json

In [ ]:
!python scripts/audit_grounding_coverage.py --map datasets/maps/shepherd_test_map_v1.csv --dataset datasets/maps/grounding_examples_v1.jsonl --dataset datasets/maps/grounding_diagnostics_v1.jsonl --dataset datasets/maps/grounding_holdout_synthetic_v1.jsonl --json-output outputs/evaluations/grounding_map_coverage_v1.json --markdown-output reports/week3_grounding_map_coverage.md

In [ ]:
!python scripts/audit_week3_completion.py --map-validation outputs/evaluations/map_validation_shepherd_test_map_v1.json --region-map-validation outputs/evaluations/map_validation_shepherd_test_map_regions_v1.json --grounding-evaluation outputs/evaluations/grounding_examples_v1.json --grounding-evaluation outputs/evaluations/grounding_diagnostics_v1.json --grounding-evaluation outputs/evaluations/grounding_holdout_synthetic_v1.json --grounding-dataset-validation outputs/evaluations/grounding_examples_v1_validation.json --grounding-dataset-validation outputs/evaluations/grounding_diagnostics_v1_validation.json --grounding-dataset-validation outputs/evaluations/grounding_holdout_synthetic_v1_validation.json --coverage-report outputs/evaluations/grounding_map_coverage_v1.json --week3-status outputs/evaluations/week3_grounding_status.json --acceptance-criteria docs/week3_acceptance_criteria.json --research-deferrals docs/week3_research_deferrals.json --json-output outputs/evaluations/week3_completion_gate_audit.json --markdown-output reports/week3_completion_gate_audit.md

In [ ]:
!python scripts/create_week3_human_grounding_packet.py --map datasets/maps/shepherd_test_map_v1.csv --jsonl-output reports/week3_human_grounding_packet.jsonl --markdown-output reports/week3_human_grounding_packet.md

In [ ]:
!python scripts/ground_intent.py --command "Inspect the greenhouse and avoid the power lines." --map datasets/maps/shepherd_test_map_v1.csv --output outputs/evaluations/grounded_intent_constraint_example.json

In [ ]:
!python scripts/ground_intent.py --command "Inspect the polygon zone." --map datasets/maps/shepherd_test_map_regions_v1.geojson --output outputs/evaluations/grounded_intent_polygon_example.json

In [ ]:
!python scripts/create_grounding_clarification_report.py --command "Monitor the road until the ambulance arrives." --map datasets/maps/shepherd_test_map_v1.csv --output outputs/evaluations/grounding_clarification_ambiguous_road.json

In [ ]:
!python scripts/apply_grounding_clarification.py --grounded-json outputs/evaluations/grounding_clarification_ambiguous_road.json --map datasets/maps/shepherd_test_map_v1.csv --choice location=loc_service_road --choice target=loc_service_road --output outputs/evaluations/grounding_resolution_ambiguous_road_service_road.json

In [ ]:
!python scripts/summarize_week3_grounding_status.py --map-validation outputs/evaluations/map_validation_shepherd_test_map_v1.json --grounding-evaluation outputs/evaluations/grounding_examples_v1.json --grounding-evaluation outputs/evaluations/grounding_diagnostics_v1.json --grounding-evaluation outputs/evaluations/grounding_holdout_synthetic_v1.json --clarification-report outputs/evaluations/grounding_clarification_ambiguous_road.json --clarification-report outputs/evaluations/grounding_clarification_unresolved_pickup.json --applied-resolution outputs/evaluations/grounding_resolution_ambiguous_road_service_road.json --output-json outputs/evaluations/week3_grounding_status.json --output-markdown reports/week3_grounding_status.md

In [ ]:
!python scripts/render_grounding_map.py --map datasets/maps/shepherd_test_map_v1.csv --command "Send two drones north and inspect the crops." --output outputs/maps/shepherd_test_map_v1_grounded_example.html

In [ ]:
!python scripts/render_grounding_map.py --map datasets/maps/shepherd_test_map_regions_v1.geojson --command "Inspect the polygon zone." --output outputs/maps/shepherd_test_map_regions_v1_polygon_example.html

The validation report is an audit artifact, not a safety certificate. The evaluations above are synthetic development, diagnostic, and holdout-style checks, not real-world grounding accuracy. The HTML map is an inspection artifact, not an evaluation metric. Use these only to verify that the explicit map, examples, parser, and deterministic grounder are internally consistent.